### Minimizing QM of MEOH and MMP together

In [9]:
import numpy as np
from scipy.optimize import minimize
from pyscf import scf, qmmm, gto
from pyscf.tools import cubegen
from pyscf.geomopt.geometric_solver import optimize
import MDAnalysis as mda
import sys; sys.path.append('/home/chemistry/QMMM/')
import visualize_QMMM as vQ
import SCF_helpers as Sh
from importlib import reload  # python 3.4+
import pickle
import os
import py3Dmol

In [10]:
# ============================================================
# Constants
# ============================================================

HARTREE_TO_KJMOL = 2625.49962

In [11]:
# Optimizing of QM MEOH
# MEOH_picklefile = "MEOH_QM.pkl"
# MEOH_cube_folder = 'cubes_optimize"
# Optimizing of QM MMP
# MMP_picklefile = "MMP_QM.pkl"

# Alternative combined optimizing 
picklefile = "MEOH_MMP_QM.pkl"
cubefolder = "cubes_optimized_MEOH_MMP"

In [33]:
#---------------- Generalized Extraction Code------------------------

# Extracting atom positions from System0.gro for both MEOH and MMP at the same time
qm_atoms, mm_coords, mm_charges, mm_names, mm_residues = Sh.read_gro(
    "../System0.gro",
    qm_residues=("MMP", "MEOH"), 
    #qm_rediues("MEOH"),  # alternative way of importing both??
    mm_residues=("NA", "SOL"),
)

# Tryng to combine the MEOH and MMP extractions, Test 1-----------------------

In [36]:
# Tester of code
# Are both MEOH and MMP being but into the opimization or just one or the other ??
display(qm_atoms)
# display(mm_names)
# display(mm_charges)


'C 30.00000000 23.30000000 52.50000000\nO 31.43000000 23.30000000 52.50000000\nH 31.74000000 24.22000000 52.50000000\nH 29.64000000 23.30000000 53.53000000\nH 29.64000000 22.41000000 51.99000000\nH 29.64000000 24.19000000 51.99000000\nP 22.45000000 31.63000000 52.50000000\nO 23.37000000 32.55000000 53.42000000\nO 23.37000000 30.71000000 51.58000000\nO 21.53000000 32.55000000 51.58000000\nO 21.53000000 30.71000000 53.42000000\nC 24.80000000 32.46000000 53.33000000\nH 25.23000000 33.46000000 53.30000000\nH 25.07000000 31.92000000 52.42000000\nH 25.18000000 31.92000000 54.20000000\nH 24.30000000 30.89000000 51.76000000'

In [38]:
# Creat a PySCF Mole object describing the uantum-mechanicall system that PsySCF will calculate

# Code form /QMMM/MMP_MEOH/QM_MMP
mol = gto.M(
    atom=qm_atoms,
    basis="6-31G",
    charge=-1,
    spin=0,
    unit="Angstrom",
    verbose=4
)

# # Code from /QMMM/MMP_MEOH/QM_MEOH
# mol = gto.M(
#     atom=qm_atoms,
#     basis="sto-3g",
#     charge=0,
#     spin=0,
#     unit="Angstrom",
#     verbose=4
# )

System: uname_result(system='Linux', node='Cas2', release='6.1.177-1-MANJARO', version='#1 SMP PREEMPT_DYNAMIC Sat Jul  4 22:18:29 UTC 2026', machine='x86_64')  Threads 32
Python 3.14.6 (main, Jun 15 2026, 11:36:54) [GCC 16.1.1 20260430]
numpy 2.5.1  scipy 1.18.0  h5py 3.16.0
Date: Fri Oct  2 09:52:45 2026
PySCF version 2.14.0
PySCF path  /home/chemistry/venvs/jupyter/lib/python3.14/site-packages/pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 16
[INPUT] num. electrons = 76
[INPUT] charge = -1
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mole.unit = Angstrom
[INPUT] Symbol           X                Y                Z      unit          X                Y                Z       unit  Magmom
[INPUT]  1 C     30.000000000000  23.300000000000  52.500000000000 AA   56.691783736952  44.030618702366  99.210621539666 Bohr   0.0
[INPUT]  2 O     31.430000000000  23.300000000000  52.500000000000 AA   59.394092095080  44.03061870

In [40]:
# SCF calculation
mf = scf.RHF(mol)

# Start timing
start_time = time.time()

# MMP Geometry optimization
# mol_opt = optimize(mf, maxsteps=10)
mol_opt = optimize(mf)

print("optimization time", time.time()-start_time)

# MEOH Geometry optimization
mol_opt = optimize(mf)

print("\nOptimized geometry:")
print(mol_opt.atom_coords(unit="Angstrom"))

NameError: name 'time' is not defined

In [ ]:
# Additional code from MEOH minimize

In [2]:
# MMP
print("\nOptimized geometry:")
print(mol_opt.atom_coords(unit="Angstrom"))


# ============================================================
# SCF calculation at optimized geometry
# ============================================================

mf_opt = scf.RHF(mol_opt)
energy = mf_opt.kernel()

print("\nOptimized SCF energy:")
print(energy)

print("\nNumber of MOs:")
print(mf_opt.mo_coeff.shape[1])

mo_coeffs = mf_opt.mo_coeff

print("\nOptimized geometry:")
print(mol_opt.atom_coords(unit="Angstrom"))

##---------------------------------------------------------------------------------------------

# MEOH
# ============================================================
# SCF calculation at optimized geometry
# ============================================================

mf_opt = scf.RHF(mol_opt)
energy = mf_opt.kernel()

print("\nOptimized SCF energy:")
print(energy)

print("\nNumber of MOs:")
print(mf_opt.mo_coeff.shape[1])

mo_coeffs = mf_opt.mo_coeff


Optimized geometry:


NameError: name 'mol_opt' is not defined

In [ ]:
# ============================================================
# Generate orbital cube files
# ============================================================

os.makedirs(cube_folder, exist_ok=True)

resolution = 0.3

norbs = mf_opt.mo_coeff.shape[1]
n_occ = sum(mf_opt.mo_occ > 0)

orbital_filenames = []

for iorb in range(norbs):

    filename = f"{cube_folder}/orbital_{iorb}.cube"

    cubegen.orbital(
        mol_opt,
        filename,
        mf_opt.mo_coeff[:, iorb],
        resolution=resolution
    )

    orbital_filenames.append(filename)

print(f"\nGenerated {norbs} orbital cube files.")
print(f"Directory: {cube_folder}")

In [ ]:
qm_data = {
    "mol_opt": mol_opt,
    "mo_coeff": mf_opt.mo_coeff,
    "mo_energy": mf_opt.mo_energy,
    "mo_occ": mf_opt.mo_occ,
    "e_tot": mf_opt.e_tot,
    "n_occ": int(sum(mf_opt.mo_occ > 0)),
    "cube_folder": cube_folder,
    "orbital_filenames": orbital_filenames,
}

with open(picklefile, "wb") as f:
    pickle.dump(qm_data, f)

print(picklefile)